In [0]:
#Load bronze data
df = spark.table("workspace.default.bronze_wikiart")
print(f"Bronze rows: {df.count()}")

Bronze rows: 80042


In [0]:
from pyspark.sql.functions import *

In [0]:
#Extract style from filename
df = df.withColumn("style", split(col("filename"), "/")[0])

In [0]:
#Remove uncertain artist rows
df = df.filter(col("subset") != "uncertain artist")
print(f"After removing uncertain artists: {df.count()}")

After removing uncertain artists: 79998


In [0]:
#Remove phash duplicates
df = df.dropDuplicates(["phash"])
print(f"After removing duplicates: {df.count()}")

After removing duplicates: 79998


In [0]:
#Clean the genre column
df = df.withColumn(
    "primary_genre",
    regexp_replace(
        split(
            regexp_replace(col("genre"), "[\\[\\]']", ""),
            ","
        )[0],
        "^ | $", ""
    )
)

In [0]:
#Trim and standardize artist names
df = df.withColumn("artist", trim(col("artist")))
df = df.withColumn("artist", regexp_replace(col("artist"), "\\s+", " "))

In [0]:
#Filter extreme dimensions
df = df.filter(
    (col("width") >= 200) & 
    (col("height") >= 200) & 
    (col("width") <= 10000) & 
    (col("height") <= 10000)
)
print(f"After dimension filter: {df.count()}")

After dimension filter: 79989


In [0]:
#Select and reorder final columns
df_silver = df.select(
    "filename",
    "style",
    "artist",
    "primary_genre",
    "description",
    "phash",
    "width",
    "height",
    "genre_count",
    "subset"
)

In [0]:
#Save as Silver table
df_silver.write.mode("overwrite").saveAsTable("workspace.default.silver_wikiart")
print(f"Silver table saved. Rows: {df_silver.count()}")

Silver table saved. Rows: 79989


In [0]:
#Final check

silver = spark.table("workspace.default.silver_wikiart")
print(f"Silver rows: {silver.count()}")
print(f"Unique styles: {silver.select('style').distinct().count()}")
print(f"Unique artists: {silver.select('artist').distinct().count()}")
print(f"Unique genres: {silver.select('primary_genre').distinct().count()}")
display(silver.limit(5))

Silver rows: 79989
Unique styles: 27
Unique artists: 1119
Unique genres: 27


filename,style,artist,primary_genre,description,phash,width,height,genre_count,subset
Abstract_Expressionism/alexander-liberman_untitled-abstract-1977.jpg,Abstract_Expressionism,alexander liberman,Abstract Expressionism,untitled-abstract-1977,b83fc744ec4c3c31,1382,1807,1,train
Abstract_Expressionism/alice-baber_green-swing.jpg,Abstract_Expressionism,alice baber,Abstract Expressionism,green-swing,9250147f8caebe74,2289,1382,1,train
Abstract_Expressionism/arshile-gorky_golden-brown-painting-1944.jpg,Abstract_Expressionism,arshile gorky,Abstract Expressionism,golden-brown-painting-1944,d3b3fcc92628d2c4,1766,1382,1,train
Abstract_Expressionism/arshile-gorky_untitled-1948.jpg,Abstract_Expressionism,arshile gorky,Abstract Expressionism,untitled-1948,e2338d48a47cd95b,1629,1382,1,train
Abstract_Expressionism/arthur-pinajian_untitled-landscape-bellport-no-224-1989.jpg,Abstract_Expressionism,arthur pinajian,Abstract Expressionism,untitled-landscape-bellport-no-224-1989,e6a0489b35e9e62d,1775,1382,1,train
